In [11]:
from common import *
if "BEST" not in globals():
    BEST = {}
    for name, (fac, grid) in GRIDS.items():
        BEST[name] = cv_tune(fac, grid)
        print(name, round(BEST[name][0], 3), BEST[name][1], BEST[name][2])

Xfit, Xcal = cal_split(norm); Xte = test[feats].to_numpy()
PRED, SCORE = {}, {}
for name, (fac, _) in GRIDS.items():
    if name not in BEST:
        BEST[name] = cv_tune(fac, GRIDS[name][1])
    _, params, q = BEST[name]
    fn = fac(Xfit, **params); t1, t2 = np.quantile(fn(Xcal), q)
    SCORE[name] = fn(Xte); PRED[name] = zones(SCORE[name], t1, t2)
joblib.dump(dict(PRED=PRED, SCORE=SCORE, BEST=BEST), RES / "unsup.pkl")


['C:\\Users\\Prisha\\Downloads\\ml_turbofan\\data preprocessing\\processed_output\\run_20261029_201358_328692\\results\\unsup.pkl']

In [12]:
from common import *
if "BEST" not in globals():
    BEST = {}
for name, (fac, grid) in GRIDS.items():
    if name not in BEST:
        BEST[name] = cv_tune(fac, grid)
    print(name, round(BEST[name][0], 3), BEST[name][1], BEST[name][2])


Mahalanobis (d=1) 0.628 {'d': 1, 'reg': 1e-06} (0.95, 0.999)
CANARI 0.646 {'d': 3, 'reg': 0.001} (0.95, 0.999)
Isolation Forest 0.684 {'n': 200} (0.95, 0.999)
One-Class SVM 0.696 {'nu': 0.02} (0.95, 0.999)
LOF 0.66 {'k': 50} (0.95, 0.999)
Autoencoder 0.471 {'h': (8, 3, 8)} (0.9, 0.99)
Hebbian (Sanger) 0.482 {'k': 3} (0.9, 0.99)


In [13]:
from common import *
print(RUN.name, len(feats), train_lab.shape, norm.shape, test.shape)
print(train_lab.label.value_counts().to_dict(), test.label.value_counts().to_dict())


run_20261029_201358_328692 14 (111318, 20) (81062, 20) (49041, 20)
{'normal': 81062, 'near_anomaly': 22320, 'anomaly': 7936} {'normal': 36048, 'near_anomaly': 9585, 'anomaly': 3408}


In [17]:
import importlib, common; importlib.reload(common)
from common import *
print(hasattr(common, "f_ens"))   # must print True

True


In [18]:
from common import *
u = joblib.load(RES / "unsup.pkl")
print(u["BEST"]["CANARI"], u["BEST"]["One-Class SVM"])   # expect (score, params, q)
pc, po = u["BEST"]["CANARI"][1], u["BEST"]["One-Class SVM"][1]
best = cv_tune(f_ens, [dict(pc=pc, po=po)])              # same CV and quantile tuning as other models
_, params, q = best
Xf, Xc = cal_split(norm)
fn = f_ens(Xf, **params); sc, st = fn(Xc), fn(test[feats].values)
t1, t2 = np.quantile(sc, q)
u["PRED"]["Ensemble"], u["SCORE"]["Ensemble"], u["BEST"]["Ensemble"] = zones(st, t1, t2), st, best
joblib.dump(u, RES / "unsup.pkl")

(np.float64(0.661424092658485), {'d': 2, 'reg': 0.1}, (0.9, 0.999)) (np.float64(0.68950046393601), {'nu': 0.02}, (0.9, 0.999))


['C:\\Users\\Prisha\\Downloads\\ml_turbofan\\data preprocessing\\processed_output\\run_20261111_135602_491576\\results\\unsup.pkl']

In [21]:
import importlib, common; importlib.reload(common)
from common import *
u = joblib.load(RES / "unsup.pkl")
Xf, _ = cal_split(norm)
fl = {}
for name, fac in [("CANARI", f_canari), ("One-Class SVM", f_ocsvm)]:
    params = u["BEST"][name][1]
    fn = fac(Xf, **params)
    fl[name] = few_label_eval(fn(train_lab[feats].values), fn(test[feats].values))
    print(name, {k: (round(a, 3), round(b, 3)) for k, (a, b) in fl[name].items()})
joblib.dump(fl, RES / "few_label.pkl")

CANARI {0.02: (np.float64(0.679), np.float64(0.02)), 0.05: (np.float64(0.685), np.float64(0.012)), 0.1: (np.float64(0.692), np.float64(0.002)), 0.2: (np.float64(0.693), np.float64(0.004))}
One-Class SVM {0.02: (np.float64(0.741), np.float64(0.005)), 0.05: (np.float64(0.744), np.float64(0.004)), 0.1: (np.float64(0.746), np.float64(0.002)), 0.2: (np.float64(0.748), np.float64(0.0))}


['C:\\Users\\Prisha\\Downloads\\ml_turbofan\\data preprocessing\\processed_output\\run_20261111_135602_491576\\results\\few_label.pkl']